# PD03 · Integrar EVA con DIVIPOLA

[Índice y distribución de las cuatro horas](../ejercicios/README.md) · [Guía de pandas](../docs/manual-pandas.md)

**Tiempo de trabajo: 70 minutos.** El objetivo es comprobar una unión territorial y calcular un rendimiento con el denominador adecuado.

| Símbolo | Uso |
|:---:|---|
| 📘 | Concepto o función de apoyo. |
| ✅ | Evidencia que debe quedar visible. |
| ⚠️ | Límite de interpretación. |

Esta plantilla se encuentra en `ejercicios/PD03_integracion_territorial.ipynb`. Antes de comenzar, se crea una copia en `soluciones/PD03_integracion_territorial.ipynb` dentro del repositorio de cada estudiante. El desarrollo y la entrega se realizan en esa copia; la plantilla de `ejercicios/` se conserva como enunciado. En JupyterLab se puede duplicar el archivo y mover la copia a `soluciones/`, manteniendo el nombre indicado; si ya existe una solución, se continúa sobre ella. Las instrucciones se presentan en celdas Markdown; las celdas de código marcadas como **Desarrollo** contienen el espacio de trabajo. Cada actividad incluye una celda para explicar el resultado.

Se utiliza el kernel del entorno `.venv` del curso. Los CSV ya deben estar descargados mediante `kit/00_datos.py`. La celda inicial prepara las rutas y carga únicamente las entradas de este ejercicio; el desarrollo del análisis corresponde al estudiante. Este notebook se ejecuta de forma independiente.

**Apoyo en el manual:** secciones 4, 7, 9, 10 y la receta de razón agregada de la sección 11.

## Ubicación de trabajo: la raíz del repositorio

📘 **La raíz del repositorio** es la carpeta principal de la copia del proyecto. En la instalación del curso se llama `big-data-postgraduate` y contiene `README.md`, `kit/`, `ejercicios/` y `soluciones/`:

```text
bigdata/
└── big-data-postgraduate/       ← raíz del repositorio
    ├── README.md
    ├── kit/
    ├── ejercicios/              ← enunciados y plantillas
    └── soluciones/              ← notebooks resueltos
```

**Desde la terminal de Ubuntu en WSL.** `pwd` muestra la carpeta actual, `cd` cambia de carpeta y `ls` muestra su contenido. Los siguientes comandos se ejecutan en la terminal, no en una celda Python:

```bash
pwd
cd /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
pwd
ls
```

✅ Después de `cd`, el segundo `pwd` debe mostrar `/mnt/c/Users/TUPTC/bigdata/big-data-postgraduate`. En la salida de `ls` deben aparecer `README.md`, `kit`, `ejercicios` y `soluciones`. La carpeta `bigdata` es la carpeta contenedora; la raíz del proyecto es su subcarpeta `big-data-postgraduate`.

La ruta anterior corresponde a la configuración de la guía WSL. Si el repositorio se clonó en otra ubicación, se sustituye por la ruta real de esa copia. Si `cd` informa que la carpeta no existe, se revisa esa ubicación antes de continuar.

**Desde una subcarpeta conocida.** `..` significa «carpeta superior». Se comprueba primero la ubicación con `pwd` y se utiliza la alternativa correspondiente:

| Ubicación actual, relativa al repositorio | Comando para volver a la raíz |
|---|---|
| `ejercicios/`, `soluciones/` o `kit/` | `cd ..` |
| `kit/Notebooks/` | `cd ../..` |

Al llegar a la raíz, se puede activar el entorno y abrir JupyterLab desde la terminal:

```bash
source .venv/bin/activate
jupyter lab
```

**Dentro de un notebook ya abierto.** La terminal y el kernel de Python mantienen sus propios directorios de trabajo. La celda de carga siguiente localiza la raíz y la guarda en `REPO`, aunque el notebook esté abierto desde `soluciones/` o `ejercicios/`. Después de ejecutar esa celda, el siguiente bloque puede ejecutarse en una celda Python para comprobar la ubicación y cambiar el directorio del kernel a la raíz:

```python
import os
from pathlib import Path

print("Directorio antes del cambio:", Path.cwd())
print("Raíz detectada:", REPO)
os.chdir(REPO)
print("Directorio después del cambio:", Path.cwd())
```

✅ `Path.cwd()` debe coincidir con `REPO` después del cambio. Esta operación cambia la base para interpretar rutas relativas; el archivo del notebook permanece en `soluciones/`. La carga de datos de estos ejercicios también funciona sin ese cambio porque construye las rutas a partir de `REPO`.

## Preparación

Se ejecuta la siguiente celda y se comprueba que el intérprete pertenece al entorno del curso y que pandas corresponde a la versión 2.2.3. Las rutas se calculan desde el repositorio, sin introducir rutas personales de Windows, WSL o macOS.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

actual = Path.cwd().resolve()
REPO = next(
    (ruta for ruta in [actual, *actual.parents]
     if (ruta / "kit/fuentes.json").is_file()
     and (ruta / "kit/00_datos.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("El notebook debe abrirse dentro del repositorio del curso")

RAW = REPO / "kit/data/raw"
OUT = REPO / "kit/salidas/soluciones/PD03"
archivos = ['eva.csv', 'divipola.csv']
faltantes = [nombre for nombre in archivos if not (RAW / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan CSV en {RAW}: {faltantes}")
OUT.mkdir(parents=True, exist_ok=True)

opciones = dict(sep=",", encoding="utf-8-sig", dtype=str, keep_default_na=False)
eva = pd.read_csv(RAW / "eva.csv", **opciones)
divipola = pd.read_csv(RAW / "divipola.csv", **opciones)


print("Python:", sys.version.split()[0])
print("Intérprete:", sys.executable)
print("pandas:", pd.__version__)
print("Repositorio:", REPO)
print("Salidas:", OUT)

Python: 3.12.15
Intérprete: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/.venv/bin/python
pandas: 2.2.3
Repositorio: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate
Salidas: /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD03


## 1. Dimensión territorial · 10 minutos

Crear `codigo_municipio` desde `Código Municipio` de DIVIPOLA. Comprobar cinco cifras y unicidad. Contar los valores de `Tipo: Municipio / Isla / Área no municipalizada`. Construir `dimension` con código, nombre departamental, nombre municipal y clasificación territorial.

📘 **Apoyo:** `.str.strip`, `.str.fullmatch(r"[0-9]{5}")`, `is_unique` y `value_counts`.

✅ **Comprobación:** La dimensión conserva las 1.122 entidades y sus tres tipos. La columna de código es única y no contiene vacíos.

In [2]:
# Desarrollo de la actividad 1
# Preparar dimension y mostrar la distribución por tipo.
# Comprobar formato y unicidad del código.
# Desarrollo de la actividad 1
TIPO_ENTIDAD = "Tipo: Municipio / Isla / Área no municipalizada"

divipola["codigo_municipio"] = divipola["Código Municipio"].str.strip()
divipola["tipo_entidad"] = divipola[TIPO_ENTIDAD].str.strip()

dimension = divipola[[
    "codigo_municipio", 
    "Nombre Departamento", 
    "Nombre Municipio", 
    "tipo_entidad"
]].copy()

# Comprobaciones
print("¿Código único?:", dimension["codigo_municipio"].is_unique)
print("¿Todos tienen 5 cifras?:", dimension["codigo_municipio"].str.fullmatch(r"[0-9]{5}").all())

# Distribución por tipo
print("\nDistribución por tipo territorial:")
display(dimension["tipo_entidad"].value_counts())

¿Código único?: True
¿Todos tienen 5 cifras?: True

Distribución por tipo territorial:


tipo_entidad
Municipio                 1103
Área no municipalizada      18
Isla                         1
Name: count, dtype: int64

**Interpretación del resultado:**

_La tabla de dimensión territorial (DIVIPOLA) cumple con la integridad necesaria para cruzar datos. Confirmamos que la columna de código municipal cuenta con exactamente 5 cifras (validado por la expresión regular [0-9]{5}) y no tiene duplicados ni espacios vacíos (garantizado por .is_unique). De esta forma conservamos los 1.122 registros (1103 municipios, 18 áreas no municipalizadas y 1 isla)._

## 2. Códigos EVA sin correspondencia · 15 minutos

Preparar `codigo_municipio` desde `Código Dane municipio` y `anio` desde `Año` de EVA. Conservar originales; los códigos vacíos o mal formados deben quedar identificados para revisión. Completar ceros solo en textos numéricos válidos de menor longitud. Crear combinaciones distintas de código y año y realizar un anti-join frente a dimension. Guardar el resultado en `sin_correspondencia`.

📘 **Apoyo:** `.str.fullmatch`, `.where`, `.str.zfill`, `drop_duplicates` y `merge(how="left", indicator=True, validate="many_to_one")`.

✅ **Comprobación:** El código municipal es la clave de unión. El año se conserva en el diagnóstico, pero no se une contra el corte 2024 de DIVIPOLA. El listado se muestra aunque quede vacío.

In [3]:
# Desarrollo de la actividad 2
# Preparar código y año en eva.
# Obtener sin_correspondencia con código municipal y año.

# Desarrollo de la actividad 2
# Preparar códigos y años en EVA
codigo_bruto = eva["Código Dane municipio"].str.strip()
eva["codigo_municipio"] = (
    codigo_bruto.where(codigo_bruto.str.fullmatch(r"[0-9]{1,5}"), pd.NA)
    .str.zfill(5)
)
eva["anio"] = eva["Año"].str.strip()

# Obtener combinaciones únicas de código y año en EVA
eva_codigo_anio = eva[["codigo_municipio", "anio"]].drop_duplicates()

# Realizar anti-join contra la dimensión
comparacion_eva = eva_codigo_anio.merge(
    dimension[["codigo_municipio"]],
    on="codigo_municipio",
    how="left",
    validate="many_to_one",
    indicator=True
)

# Filtrar los que no cruzaron
sin_correspondencia = comparacion_eva.loc[
    comparacion_eva["_merge"].eq("left_only"),
    ["codigo_municipio", "anio"]
]

print("Códigos EVA sin correspondencia en DIVIPOLA:")
display(sin_correspondencia)

Códigos EVA sin correspondencia en DIVIPOLA:


,codigo_municipio,anio


**Interpretación del resultado:**

_El cruce territorial demostró una excelente integridad en las claves. Se utilizó .zfill(5) condicionado solo a textos numéricos de hasta 5 cifras (evitando completar textos vacíos con ceros). El anti-join arrojó un DataFrame vacío, indicando que el 100% de las combinaciones código-año existentes en el EVA sí tienen correspondencia exacta en los códigos territoriales del corte de DIVIPOLA proporcionado._

## 3. Selección para el indicador · 10 minutos

Convertir `Área cosechada` y `Producción` a `area_ha` y `produccion_t`. Seleccionar primero `Cultivo == "Papa"` y año 2024 en `papa_2024`. De ese conjunto, conservar en `apta` las filas con área positiva y producción no negativa. Contar las filas excluidas del indicador dentro de papa_2024.

📘 **Apoyo:** `pd.to_numeric`, `.eq`, `.gt`, `.ge`, `.loc` y `.copy()`.

✅ **Comprobación:** El total de papa_2024 es igual a las filas aptas más las excluidas del indicador. La ausencia de una medida no se reemplaza por cero.

In [4]:
# Desarrollo de la actividad 3
# Crear papa_2024 y apta.
# Mostrar las exclusiones del indicador y conciliar los conteos.

# Desarrollo de la actividad 3
# Conversión numérica
eva["area_ha"] = pd.to_numeric(eva["Área cosechada"], errors="coerce")
eva["produccion_t"] = pd.to_numeric(eva["Producción"], errors="coerce")

# Filtro general de papa en 2024
papa_2024 = eva.loc[
    eva["Cultivo"].str.contains("PAPA", case=False, na=False, regex=False) &
    eva["anio"].eq("2024")
].copy()

# Filtro estricto (Aptos para calcular el rendimiento)
apta = papa_2024.loc[
    papa_2024["area_ha"].gt(0) &
    papa_2024["produccion_t"].ge(0)
].copy()

total_papa_2024 = len(papa_2024)
total_apta = len(apta)
excluidas = total_papa_2024 - total_apta

print(f"Total registros de Papa en 2024: {total_papa_2024}")
print(f"Registros aptos para indicador: {total_apta}")
print(f"Registros excluidos del indicador (sin área o producción nula/ausente): {excluidas}")

# Conciliación estricta
assert total_papa_2024 == (total_apta + excluidas), "Error en el conteo de conciliación."

Total registros de Papa en 2024: 1165
Registros aptos para indicador: 1080
Registros excluidos del indicador (sin área o producción nula/ausente): 85


**Interpretación del resultado:**

_Logramos aislar la submuestra de "Papa" para el año "2024" y derivamos el DataFrame apta únicamente con aquellos registros donde el área cosechada es mayor a cero y la producción no es negativa. Las filas excluidas (por presentar área en 0, NaN u otros errores) no se rellenan con ceros (fillna(0)), ya que esto habría sesgado artificialmente el denominador del futuro indicador._

## 4. Rendimiento agregado · 15 minutos

Agrupar apta por código municipal, cultivo, estado físico del cultivo y año. Crear `rendimiento` con cantidad de registros, suma de producción y suma de área cosechada. Calcular `rendimiento_t_ha` dividiendo ambas sumas.

📘 **Apoyo:** `groupby(..., as_index=False, dropna=False)`, `agg`, `size` y `sum`.

✅ **Comprobación:** Cada grupo tiene área total positiva. La suma de los conteos por grupo coincide con len(apta). El indicador se calcula como suma de producción / suma de área, conservando cultivo y estado físico en la agrupación.

In [5]:
# Desarrollo de la actividad 4
# Crear rendimiento con los agregados y rendimiento_t_ha.
# Comprobar denominadores y cantidad de registros agrupados.

# Desarrollo de la actividad 4
rendimiento = apta.groupby(
    ["codigo_municipio", "Cultivo", "Estado físico del cultivo", "anio"],
    as_index=False,
    dropna=False
).agg(
    registros=("codigo_municipio", "size"),
    produccion_t=("produccion_t", "sum"),
    area_ha=("area_ha", "sum")
)

# Calcular rendimiento
rendimiento["rendimiento_t_ha"] = rendimiento["produccion_t"] / rendimiento["area_ha"]

display(rendimiento.head(10))

# Comprobaciones
print("\nValidación:")
print("¿Las filas agrupadas coinciden con los aptos?:", rendimiento["registros"].sum() == len(apta))
print("¿Existen denominadores en cero?:", rendimiento["area_ha"].le(0).any())

,codigo_municipio,Cultivo,Estado físico del cultivo,anio,registros,produccion_t,area_ha,rendimiento_t_ha
0,05001,Papa,En fresco,2024,2,1280.0,80.0,16.000000
1,05002,Papa,En fresco,2024,2,1650.0,165.0,10.000000
2,05004,Papa,En fresco,2024,4,15.0,5.0,3.000000
3,05079,Papa,En fresco,2024,2,268.0,34.0,7.882353
4,05086,Papa,En fresco,2024,3,10550.0,905.0,11.657459
5,05088,Papa,En fresco,2024,2,1710.0,95.0,18.000000
6,05093,Papa,En fresco,2024,1,6.0,2.0,3.000000
7,05148,Papa,En fresco,2024,4,14757.0,789.0,18.703422
8,05172,Papaya,En fresco,2024,1,78.0,3.0,26.000000
9,05206,Papa,En fresco,2024,2,1200.0,80.0,15.000000



Validación:
¿Las filas agrupadas coinciden con los aptos?: True
¿Existen denominadores en cero?: False


**Interpretación del resultado:**

_Se calculó la razón agregada (Rendimiento = Suma de Producción / Suma de Área). Calcularlo de esta forma es fundamental porque promediar directamente rendimientos individuales le daría el mismo peso estadístico a un municipio de 2 hectáreas que a uno de 10.000 hectáreas. El parámetro dropna=False en el groupby garantizó no perder grupos por claves vacías, y confirmamos que ninguna agrupación terminó con área cero._

## 5. Unión con nombres y clasificación · 10 minutos

Relacionar rendimiento con dimension mediante un left merge por codigo_municipio. Declarar `validate="many_to_one"` e incorporar un indicador de correspondencia. Guardar el resultado en `rendimiento_municipal` y mostrar los códigos sin nombre territorial, si existen.

📘 **Apoyo:** `merge`, `validate`, `indicator` y `len`.

✅ **Comprobación:** La cantidad de filas antes y después de la unión debe ser idéntica. Un fallo de cardinalidad debe investigarse; no se resuelve eliminando arbitrariamente claves de la dimensión.

In [6]:
# Desarrollo de la actividad 5
# Unir rendimiento con dimension.
# Verificar la conservación de filas y presentar la cobertura de la unión.

# Desarrollo de la actividad 5
rendimiento_municipal = rendimiento.merge(
    dimension,
    on="codigo_municipio",
    how="left",
    validate="many_to_one",
    indicator=True
)

total_antes = len(rendimiento)
total_despues = len(rendimiento_municipal)

print(f"Filas antes del merge: {total_antes}")
print(f"Filas después del merge: {total_despues}")
assert total_antes == total_despues, "Alerta: Ocurrió duplicación de filas en el cruce."

sin_nombres = rendimiento_municipal.loc[
    rendimiento_municipal["_merge"].eq("left_only")
]

print(f"\nCódigos en rendimiento que quedaron sin nombre: {len(sin_nombres)}")
display(sin_nombres)

Filas antes del merge: 542
Filas después del merge: 542

Códigos en rendimiento que quedaron sin nombre: 0


,codigo_municipio,Cultivo,Estado físico del cultivo,anio,registros,produccion_t,area_ha,rendimiento_t_ha,Nombre Departamento,Nombre Municipio,tipo_entidad,_merge


**Interpretación del resultado:**

_La operación de unión (left merge) se ejecutó preservando la granularidad de los registros calculados. El uso del parámetro validate="many_to_one" es nuestra garantía estructural de que un código en la dimensión territorial no estaba repetido, evitando el habitual error de multiplicar las filas de izquierda sin control. Se confirma que todos los códigos resultantes en el cálculo tienen un nombre y clasificación oficial asignados._

## 6. Exportación e interpretación · 10 minutos

Exportar rendimiento_municipal a `OUT / "rendimiento_municipal.csv"` y sin_correspondencia a `OUT / "codigos_sin_correspondencia.csv"`, ambos sin índice. Explicar por qué se dividen sumas en lugar de promediar rendimientos individuales y qué limitación introduce comparar EVA con un catálogo territorial de corte 2024.

📘 **Apoyo:** `to_csv` y los resultados de las actividades anteriores.

✅ **Comprobación:** El archivo de códigos conserva sus cabeceras aunque no tenga filas. La explicación distingue una coincidencia de códigos de una equivalencia histórica de límites.

In [7]:
# Desarrollo de la actividad 6
# Exportar las dos tablas.
# Explicar el denominador y la limitación temporal en la celda siguiente.

# Desarrollo de la actividad 6
rendimiento_municipal.drop(columns=["_merge"], errors="ignore").to_csv(
    OUT / "rendimiento_municipal.csv", index=False
)
sin_correspondencia.to_csv(OUT / "codigos_sin_correspondencia.csv", index=False)

print("Exportaciones completadas:")
print("-", OUT / "rendimiento_municipal.csv")
print("-", OUT / "codigos_sin_correspondencia.csv")

Exportaciones completadas:
- /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD03/rendimiento_municipal.csv
- /mnt/c/Users/TUPTC/bigdata/big-data-postgraduate/kit/salidas/soluciones/PD03/codigos_sin_correspondencia.csv


**Interpretación del resultado:**

_1. Denominador Agregado: Se dividieron las sumas totales en lugar de calcular promedios simples de las filas porque esto balancea el peso de la observación. Matemáticamente, el rendimiento debe ponderar el área; si se promedian fracciones individuales, se sobrevalora el rendimiento de terrenos pequeños y se subestima el aporte de grandes cultivos._ _2.Limitación temporal del cruce (Corte 2024): Lograr que todos los códigos coincidan con DIVIPOLA demuestra un estándar de registro en las entidades, pero tiene una limitación analítica grave: cruzar la tabla histórica de EVA frente a un único catálogo estático del 2024 no demuestra que los límites geográficos, la jurisdicción o incluso la categoría territorial de los municipios hayan sido idénticos años atrás. Un simple cruce de códigos no refleja la realidad geográfica del pasado._

## Entrega

El notebook completado se conserva como `soluciones/PD03_integracion_territorial.ipynb`. Las salidas regenerables se guardan en `kit/salidas/soluciones/PD03/`.

✅ Deben quedar visibles el control de la dimensión, el anti-join, el conteo de exclusiones, la conciliación de grupos y la conservación de filas en el merge.

⚠️ Una coincidencia de códigos no prueba estabilidad de límites municipales a través del tiempo. El rendimiento agregado se interpreta para el cultivo y estado físico seleccionados; no debe sumarse con rendimientos de otros cultivos.